# Lesson 2 · Catching the air's trembling in an array — sampling and your first audible sine wave

**Learning goals**
1. Understand sampling: continuous sound → numpy array → sound you can play back again
2. Implement `samples_count`: N = duration × sample_rate
3. Implement `make_time_axis`: build the time axis
4. Read the array shape of a 1D signal: know how `(N,)` differs from a matrix `(M, N)` (the interlude also has a 30-second question to answer out loud)
5. Implement `make_sine`: generate a sine wave and play it
6. Implement `signal_summary`: summarize a signal's statistics

> These four functions are the course's "atomic operations" — every later notebook calls them, directly or indirectly.

← **Previous lesson** · [L01 · No black boxes (motivation and roadmap)](L01_motivation.en.ipynb)

> Last lesson set up **Aurora's motivation, the 6-month roadmap, and the environment health check** — why we write everything from scratch, and where we're headed next.
> This lesson moves on to **the digital representation of sound**: sampling, the time axis, and your first audible sine wave.

In [ ]:
# Shared imports (run this cell before running the whole notebook)
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import Audio, display
except ImportError:
    pass

# ── Chinese font setup: keeps Chinese plot titles/axis labels from showing up as boxes ("tofu") ──
# matplotlib's default font, DejaVu Sans, can't render Chinese. The code below picks, from a list of
# common Chinese fonts, one that is actually installed on your computer; if Chinese text in a plot still shows as □□□,
# just add the name of a Chinese font on your system to the front of the list (common on Mac: PingFang SC; on Windows:
# SimHei / Microsoft YaHei; on Linux: Noto Sans CJK SC).
from matplotlib import font_manager as _fm
_zh_candidates = ['Source Han Sans CN', 'Source Han Sans CN Normal', 'Arial Unicode MS',
                  'Heiti SC', 'SimHei', 'Microsoft YaHei', 'Noto Sans CJK SC', 'WenQuanYi Micro Hei']
_installed = {f.name for f in _fm.fontManager.ttflist}
for _name in _zh_candidates:
    if _name in _installed:
        plt.rcParams['font.sans-serif'] = [_name] + plt.rcParams['font.sans-serif']
        break
plt.rcParams['axes.unicode_minus'] = False  # while we're at it, make minus signs on the axes display correctly


## How to learn in this lesson: work it out by hand first, then let the code check you

The core method of the Aurora course isn't "run it and look at the output." It's this:

```
① Take the formula and derive it first, on paper or in Markdown
② Work out the result by hand on a tiny example (e.g. duration=0.5s, sr=8Hz → N=4)
③ Write the code
④ Run the asserts and let the code check your hand-worked result
```

**Why do it this way?**
In an interview there's no terminal, only a whiteboard. Practicing "by hand first"
trains you to derive the result yourself when all you have is a whiteboard and no code.

Every task in this lesson starts with a derivation table; fill it in before you write the code.

## 1. The physics of sound

Have you ever noticed that when you pluck a guitar string, the air trembles?

At heart, sound is a **fluctuation in air pressure**. When a guitar string vibrates, it first pushes the nearby air molecules aside; the highs and lows of pressure ripple outward through the air, layer by layer, until they hit your eardrum — and you hear a sound. A microphone does the same thing, except that it translates that wave in the air into a wave of voltage.

But a "rise and fall" can't be stored in a computer directly. Computers only understand numbers.

So we invented **sampling**: at fixed time intervals, measure the voltage once and write it down as a number. That "how often we measure" is the **sample rate** (sr), in measurements per second (Hz).

CD quality uses 44100 Hz — 44,100 measurements every second. The highest frequency the human ear can hear is about 20,000 Hz, and according to the Nyquist theorem, a sample rate of just 2 times the highest frequency is enough to reconstruct the sound completely. 44100 is **slightly above** $2 \times 20\,\mathrm{kHz} = 40\,\mathrm{kHz}$: it meets the physical lower bound for "not misreading the wave," and it leaves a little transition band for the analog anti-aliasing filter — a physical requirement, with a historical coincidence (coming up shortly) layered on top.

```
sample rate sr = 44100 Hz
duration = 1 second
number of samples N = round(duration × sr) = 44100 numbers
```

One second of audio becomes 44,100 sample values inside the computer. That is the essence of "digital sound."

---

### Follow-up 1: why exactly "2 times"? (intuition first; L34 runs the experiment)

The line "according to the Nyquist theorem" above sounds like a decree handed down from on high — hold on, let's translate it into plain language.

A wave has one peak and one trough per period. To avoid "missing" a period, **you need at least 2 points per period**: one to catch the upper half and one to catch the lower half. Do the conversion: if the wave goes around f times per second, you need to measure at least 2f times per second — that's where "2 times" comes from.

What if you measure more sparsely than that? A fast wave and a slow wave can **pass through exactly the same sample points**. The computer gets identical numbers and simply can't tell which wave you recorded — this is called **aliasing**. At 1.5 times you'll miss things (some periods only get a point and a bit); 3 times is of course safer, just more storage-hungry; 2 times is the minimum bar for "not misreading."

For now, just build the intuition: **"at least 2 points per period."** The full experiment (seeing with your own eyes two different waves pass through the same set of points, and hearing a high tone "impersonate" a low one) is in **L34 (aliasing)**; a demo cell later in this lesson gives you a sneak peek.

### Follow-up 2: why 44100 and not a nice round 44000? (optional reading; the self-review at the end comes back to it)

**The three-line version**: by the "anything > 40000 will do" logic, 44000 or 45000 would work just as well — the odd-looking 44100 is mainly a historical leftover pieced together in **the CD era to accommodate video recorder tape / TV line rates**, not "an optimal integer someone calculated." The self-review at the end comes back to this; if you want the full story, expand the fold below.

<details>
<summary><b>Expand: the full story of how 44100 grew out of videotape</b></summary>

Before the CD, digital recording had no dedicated storage medium, so engineers borrowed **video recorder tape** to store audio data. To fit the sample values neatly into the "lines" of a TV signal, the sample rate had to be an integer multiple of the TV standard's parameters. As luck would have it, the two major TV standards produced the same number:

- American NTSC: 60 fields/s × 245 lines × 3 samples stored per line = **44100**
- European PAL: 50 fields/s × 294 lines × 3 samples stored per line = **44100**

So 44100 was written into the CD standard, and it's still with us today. It isn't a calculated "optimal value"; it's a "good-enough value" that came from accommodating old equipment. (By the way: 44100 = 2²×3²×5²×7², which has lots of factors, so integer-ratio downsampling is convenient too.)

</details>

### Follow-up 3: the two "Hz" aren't the same thing

Soon you'll run into phrases like "a 1 Hz sine wave at an 8 Hz sample rate" — two Hz side by side, easy to muddle. They measure two different things:

- **The Hz of the sample rate**: how many times per second you **measure** — how many times per second the camera shutter clicks
- **The Hz of the frequency**: how many cycles per second the sound wave **vibrates** — how fast the thing being photographed is spinning

One is "the speed of observing," the other is "the speed of the wave itself." Both units happen to be called Hz (times per second), but they're about completely different things.

## ✏️ Task 0: work out N by hand first, then implement `samples_count`

**Step 1: fill in the table by hand** (don't run any code; compute with `N = round(duration × sr)`)

| duration (s) | sample_rate (Hz) | N (by hand) |
|---|---|---|
| 0.5 | 8 | ✏️ ? |
| 1.0 | 16000 | ✏️ ? |
| 0.25 | 44100 | ✏️ ? |

**Step 2**: once the table is filled in, move on and implement the function.

In [ ]:
def samples_count(duration, sample_rate):
    """Return the total number of samples N for a given duration and sample rate."""
    # ✏️ TODO: return round(duration * sample_rate)
    raise NotImplementedError("TODO: return round(duration * sample_rate)")

In [ ]:
try:
    # Check with code — does it match your hand-worked result?
    assert samples_count(0.5, 8)     == 4,     f'expected 4, got {samples_count(0.5, 8)}'
    assert samples_count(1.0, 16000) == 16000, f'expected 16000, got {samples_count(1.0, 16000)}'
    assert samples_count(0.25, 44100)== round(0.25 * 44100)
    assert samples_count(1.0, 10)    == 10

    # Print the comparison table
    print(f'  duration=0.5s,  sr=8     → N={samples_count(0.5, 8)}')
    print(f'  duration=1.0s,  sr=16000 → N={samples_count(1.0, 16000)}')
    print(f'  duration=0.25s, sr=44100 → N={samples_count(0.25, 44100)}')
    print('samples_count  OK')
except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO function above first, then run this demo cell')

In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt

    # Sample 0.5 s at an 8 Hz sample rate: only 4 points, so you can check each one
    sr, duration = 8, 0.5
    N = samples_count(duration, sr)
    t = np.arange(N) / sr
    x = np.sin(2 * np.pi * 1.0 * t)

    print(f'N = {N} points ({duration}s × {sr}Hz)')
    print(f't = {t}  (seconds)')
    print(f'x = {np.round(x, 3)}  (sample values)')

    # First lay down the "continuous" original waveform underneath (approximated with very dense points),
    # so you can see which wave the 4 sample points were "picked" from
    t_dense = np.linspace(0, duration, 400)
    plt.plot(t_dense, np.sin(2 * np.pi * 1.0 * t_dense),
             color='0.75', lw=1.2, label='original continuous waveform (the real vibration of the air)')
    plt.stem(t, x, markerfmt='C0o', linefmt='C0-', basefmt='k-', label='sample points (the numbers stored in the computer)')
    plt.xlabel('Time (s)'); plt.ylabel('Amplitude')
    plt.title(f'A 1 Hz sine wave at an 8 Hz sample rate (N={N} points)')
    plt.legend(loc='upper right', fontsize=8)
    plt.tight_layout(); plt.show()
except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO function above first, then run this demo cell')

### A preview of L34: the "misread wave" from §1 — plotted below

Follow-up 1 in §1 already gave you the intuition. Instead of repeating the definition, let's just **plot** it: the sample rate stays at 8 Hz, and we draw a 1 Hz slow wave together with a 9 Hz fast wave — 9 Hz is way past the 8/2 = 4 Hz "safety line."

**Predict before you run**: will these two waves take the same values at the 8 sample times?

(This cell doesn't depend on the functions you implement, so you can run it any time. The full listening experiment is in **L34 aliasing**.)

In [ ]:
# Aliasing preview: two completely different waves passing through the same set of sample points
import numpy as np
import matplotlib.pyplot as plt

sr = 8                        # only 8 measurements per second
t_pts = np.arange(8) / sr     # the 8 sample times within 1 second

t_dense = np.linspace(0, 1, 1000)             # dense points to approximate a "continuous" curve
slow = np.sin(2 * np.pi * 1 * t_dense)        # 1 Hz slow wave (inside the safety line)
fast = np.sin(2 * np.pi * 9 * t_dense)        # 9 Hz fast wave (way past sr/2 = 4 Hz!)

plt.figure(figsize=(9, 3.2))
plt.plot(t_dense, slow, 'C0', lw=1.5, label='1 Hz slow wave')
plt.plot(t_dense, fast, 'C1', lw=0.8, alpha=0.7, label='9 Hz fast wave')
plt.plot(t_pts, np.sin(2 * np.pi * 1 * t_pts), 'ko', ms=7, label='points sampled at 8 Hz')
plt.xlabel('Time (s)'); plt.ylabel('Amplitude')
plt.title('Two different waves through the same sample points: with too low a sample rate, the computer cannot tell them apart')
plt.legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()

# Numerical check: at these 8 times the 9 Hz fast wave takes exactly the same values as the 1 Hz slow wave
print('1 Hz values at the sample points:', np.round(np.sin(2 * np.pi * 1 * t_pts), 4))
print('9 Hz values at the sample points:', np.round(np.sin(2 * np.pi * 9 * t_pts), 4))
print('→ Identical! From these 8 numbers alone, there is no way to know which wave it was.')
print('→ That is why you need "at least 2 samples per period". The full experiment is in L34 (aliasing).')

## ✏️ Task 1: implement `make_time_axis(duration, sample_rate)`

**Step 1: work it out by hand** (duration=0.5, sr=8)

| Step | Formula | Result |
|---|---|---|
| 1. Compute N | `samples_count(0.5, 8)` | ✏️ ? |
| 2. Generate the indices n | `np.arange(N)` | ✏️ [?, ?, ?, ?] |
| 3. Convert to time t | `n / sample_rate` | ✏️ [?, ?, ?, ?] |

**Line of reasoning** (look here only after you've done it by hand):
1. N = `samples_count(duration, sample_rate)`
2. indices n = `np.arange(N)` → `[0, 1, 2, ..., N-1]`
3. time axis t = n / sample_rate

Returns: a numpy array with dtype=float64 whose first element is 0 and whose last element is < duration.

---

**Wait — why is "the last element < duration"? For a 0.5-second signal, why don't we measure the point at t=0.5?**

That's not a point we forgot; it's a deliberate design choice called "**include the start, exclude the end**." There are two reasons:

1. **No duplicates when you join clips.** Imagine two 0.5-second clips joined end to end: the second clip's first point is its own t=0, which lands at 0.5 s overall. If the first clip also measured t=0.5, the same moment would be recorded twice. With the rule "each clip only covers the half-open interval [0, duration)," the clips fit together seamlessly, with nothing extra and nothing missing.
2. **Clean point counts.** With the start included and the end excluded, the count is exactly N = duration × sr; if both ends were included it would become N+1, and every later "one second = 44,100 numbers" calculation would carry an extra one.

An analogy: cut a 0.5-meter carrot every 0.125 m and you get 4 pieces. What we count is each piece's **starting point** — 0, 0.125, 0.25, 0.375, exactly 4 of them — so the last starting point is always less than 0.5. And t=0.5 is the 0th starting point of the **next** piece; it belongs to that one.

**A common pitfall (defusing it in advance)**: many people's first instinct is "spread N points evenly from 0 to 0.5," written as `np.linspace(0, duration, N)` — that includes both ends and gives [0, 0.167, 0.333, 0.5], so the spacing between points becomes duration/(N-1) instead of 1/sr, and the assertions below will fail. Follow the line of reasoning and use `np.arange(N) / sample_rate`.

In [ ]:
def make_time_axis(duration, sample_rate):
    # ✏️ TODO: implement it with samples_count + np.arange
    raise NotImplementedError("TODO: implement it with samples_count + np.arange")

In [ ]:
try:
    import numpy as np

    t = make_time_axis(0.5, 8)
    assert isinstance(t, np.ndarray),          't must be a numpy array'
    assert t.shape == (4,),                    f'shape should be (4,), got {t.shape}'
    assert np.allclose(t, [0.0, 0.125, 0.25, 0.375]), f'wrong values: {t}'

    t2 = make_time_axis(1.0, 16000)
    assert t2.shape == (16000,)
    assert abs(t2[0]) < 1e-12,   'the first time should be 0'
    assert t2[-1] < 1.0,         'the last time should be < duration (end point excluded)'

    print('make_time_axis  OK')
    print(f'  t[:4] = {t}')
except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO function above first, then run this demo cell')

## Interlude: array shape

Imagine you've booked a movie ticket — there are two things you need to know: the **row** and the **seat number**.
NumPy arrays are the same. `shape` is the "ticket stub": it tells you how many rows the array has and how many items are in each row.

| Analogy | NumPy | Meaning |
|---|---|---|
| A line of numbers (one row of seats) | shape `(N,)` | 1D, N elements |
| A matrix (several rows of seats) | shape `(M, N)` | 2D, M rows by N columns |
| A scalar (a VIP ticket, no row at all) | shape `()` | 0D, just a single number |

An audio signal is usually `shape (N,)` — a single row of N sample values.
Later on, STFT output will be `(n_frames, n_fft//2+1)` — now it's a matrix, with time along the rows and frequency along the columns.

Get shape straight and you won't get lost in matrix multiplication.

**A 30-second question to answer out loud (no code needed)**: which of `(16000,)` and `(1, 16000)` is more like "one mono waveform"?
Hint: the first is a single row of 16,000 sample points; the second is the matrix form "1 row × 16,000 columns" — they can hold the same number of values, but their **dimensions are a different kind of thing**, and later, when you work with matrices and batches of data, you'll need to recognize which one you're holding.

In [ ]:
try:
    import numpy as np

    scalar = np.float64(3.14)
    vec    = np.array([0., 0.5, 1.])
    mat    = np.array([[1, 2], [3, 4]])

    for name, arr in [('scalar', scalar), ('vector', vec), ('matrix', mat)]:
        print(f'  {name:4s}  shape={str(arr.shape):10s}  ndim={arr.ndim}  dtype={arr.dtype}')

    # The shape of an audio signal
    t = make_time_axis(1.0, 16000)
    print(f'\n  audio time axis  shape={t.shape}  → 1D, {len(t)} points')
except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO function above first, then run this demo cell')

## ✏️ Task 2: implement `make_sine(duration, sample_rate, frequency, amplitude=1.0)`

**First, a bridge: where does the 2π in the formula come from? (Don't memorize it — derive it once)**

Think of a sine wave as a cabin on a Ferris wheel: the cabin goes around at a steady speed, and `sin` reads off its **height** at each moment. In math, **one full turn** is 2π radians — that's the official length of sin's "one turn," and sin repeats itself every 2π.

Now derive the formula in three steps:

1. `frequency = f` means the cabin **goes around f times per second**;
2. so by time t it has made f × t turns in total;
3. each turn is 2π radians, so the total angle is θ = 2π × f × t radians.

Feed that angle to sin, multiply by the amplitude A (the radius of the Ferris wheel), and you get:

**Formula**: `x(t) = amplitude × sin(2π × frequency × t)`

Sanity-check the frequency: as t goes from 0 to 1 second, θ goes from 0 to 2πf, completing exactly f full periods — exactly "f vibrations per second." So 2π isn't decoration; it's the "**turns → radians**" conversion factor. Drop it, and "f turns per second" becomes "f/(2π) turns per second," and the frequency is all wrong.

**Radian first-aid kit (read this before filling in the table)**: radians and degrees convert as **π rad = 180°**. The decimals in the θ column below look scary, but they're all special angles you memorized in high school — whatever you do, don't treat 0.785 as "0.785 degrees" (sin 0.785° ≈ 0.014, which would be off from the answer by a factor of about fifty):

| θ (radians, as a decimal) | = what fraction of π | = how many degrees | sin θ |
|---|---|---|---|
| 0.000 | 0 | 0° | 0 |
| 0.785 | π/4 | 45° | √2/2 ≈ 0.71 |
| 1.571 | π/2 | 90° | 1 |
| 2.356 | 3π/4 | 135° | √2/2 ≈ 0.71 |

**Step 1: work it out by hand** (duration=0.5s, sr=8Hz, freq=1Hz, A=1)

| n | t = n/sr (s) | θ = 2π·f·t (rad) | θ converted (**already done; you only fill in the sin column**) | sin(θ) (to 2 decimal places) |
|---|---|---|---|---|
| 0 | 0.000 | 0.000 | 0 = 0° | ✏️ ? |
| 1 | 0.125 | 0.785 | π/4 = 45° | ✏️ ? |
| 2 | 0.250 | 1.571 | π/2 = 90° | ✏️ ? |
| 3 | 0.375 | 2.356 | 3π/4 = 135° | ✏️ ? |

**Line of reasoning**:
1. Use `make_time_axis` to build the time axis t
2. Return `amplitude × np.sin(2 × np.pi × frequency × t)`

In [ ]:
def make_sine(duration, sample_rate, frequency, amplitude=1.0):
    # ✏️ TODO: implement it with make_time_axis and np.sin
    raise NotImplementedError("TODO: implement it with make_time_axis and np.sin")

In [ ]:
try:
    import numpy as np

    # Basic checks
    x = make_sine(1.0, 16, 2.0)
    assert isinstance(x, np.ndarray)
    assert x.shape == (16,)
    assert abs(x.max() - 1.0) < 1e-9,   f'max should be 1.0, got {x.max()}'
    assert abs(x.min() + 1.0) < 1e-9,   f'min should be -1.0, got {x.min()}'

    # Amplitude check
    x2 = make_sine(1.0, 16, 2.0, amplitude=3.0)
    assert abs(x2.max() - 3.0) < 1e-9,  f'with amplitude=3, max should be 3.0'

    # Matches the numpy reference
    t = make_time_axis(1.0, 16)
    ref = np.sin(2 * np.pi * 2.0 * t)
    assert np.allclose(x, ref), 'result does not match the np.sin reference'

    # Compare with your hand calculation: duration=0.5, sr=8, freq=1
    x_check = make_sine(0.5, 8, 1.0)
    print('Hand-calculation check (freq=1Hz, sr=8Hz, duration=0.5s):')
    print(f'  code output = {np.round(x_check, 3)}')
    print(f'  reference   = {np.round(np.sin([0, np.pi/4, np.pi/2, 3*np.pi/4]), 3)}')
    print('make_sine  OK')
except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO function above first, then run this demo cell')

In [ ]:
import matplotlib.pyplot as plt

sr, dur = 200, 1.0
try:
    t = make_time_axis(dur, sr)
    fig, axes = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
    for ax, freq, color in zip(axes, [2, 5, 10], ['C0', 'C1', 'C2']):
        ax.plot(t, make_sine(dur, sr, freq), color=color)
        ax.set_ylabel(f'{freq} Hz'); ax.set_ylim(-1.3, 1.3); ax.grid(alpha=0.3)
    axes[-1].set_xlabel('Time (s)')
    fig.suptitle('The higher the frequency, the more peaks per unit of time')
    plt.tight_layout(); plt.show()
except (NotImplementedError, TypeError):
    print('⬜ Finish make_time_axis / make_sine first, then run this demo cell')
else:
    try:
        from IPython.display import Audio, display
        x_audio = make_sine(1.5, 16000, 440)
        print('Playing 440 Hz (the A4 note) — you will hear the standard "la"')
        display(Audio(x_audio, rate=16000))
    except Exception:
        print('(Not a Jupyter environment; skipping playback)')

## Parameter experiment: frequency and pitch

**Prediction questions (answer first, then run to check)**:
- What's the maximum value of `make_sine(1.0, 16000, 880)`? (Ignoring floating-point fuzz: the theoretical maximum = amplitude, which defaults to 1.0)
- Is its shape the same as that of `make_sine(1.0, 16000, 440)`?
- Which frequency sounds higher in pitch?

Pitch is set by frequency. Doubling the frequency = going up one octave:

| Note | Frequency |
|---|---|
| A3 (an octave lower) | 220 Hz |
| A4 (the standard "la") | 440 Hz |
| A5 (an octave higher) | 880 Hz |

In [ ]:
try:
    from IPython.display import Audio, display
    sr = 16000
    for name, freq in [('A3 (220 Hz)', 220), ('A4 (440 Hz)', 440), ('A5 (880 Hz)', 880)]:
        wave = make_sine(1.0, sr, freq)
        print(f'{name}  shape={wave.shape}  max={wave.max():.3f}')
        display(Audio(wave, rate=sr))
except (NotImplementedError, TypeError):
    print('⬜ Finish make_sine first, then run this cell')
except Exception:
    sr = 16000
    for name, freq in [('A3', 220), ('A4', 440), ('A5', 880)]:
        wave = make_sine(1.0, sr, freq)
        print(f'{name}  shape={wave.shape}  max={wave.max():.3f}')

## Groundwork: what does RMS actually measure? Why not just take the average?

The `signal_summary` you're about to implement has a new face in it: **RMS (Root Mean Square)**. Plain language first, then the formula.

**The question it's trying to answer: how loud is this sound "on average"?**

The first instinct is to take the mean — but a sine wave spends half its time positive and half negative, so when you add it all up the positives and negatives cancel and mean ≈ 0. Music at 100 decibels and total silence both have a mean of 0; that ruler can't measure loudness. Loudness cares about "how far from 0," not whether it's positive or negative.

So we go in three steps: first **square** (this turns negatives positive, and boosts the contribution of large amplitudes along the way), then take the **mean**, and finally take the **root** (converting the unit back from "amplitude squared" to "amplitude"). As a formula:

RMS = √( mean(x²) )

The name Root-Mean-Square is just those three steps read backward. RMS roughly corresponds to the **average loudness** your ear perceives, and in engineering it's also commonly used as a measure of energy; L40 (spectrum) and L51 (real audio) both use it as a ruler.

**So where does the hint "for a pure sine wave, RMS = A/√2" come from?**

The key step: look at **what sin² looks like** (the next cell plots it). sin swings between -1 and +1; once squared, sin² swings between 0 and 1, and its shape is **symmetric about the height 1/2** — the "hills" above 1/2 and the "valleys" below 1/2 have the same area. So even without integrals you can see that **the average of sin² is exactly 1/2**. Plug that into the RMS formula:

RMS = √( mean(A²·sin²) ) = √( A² × 1/2 ) = A/√2 ≈ 0.707 × A

That's where the √2 comes from — it isn't a magic number; it's "the average of sin² is one half," square-rooted. (A rigorous proof uses integrals, which later lessons will get to; but the picture of "symmetric above and below, equal areas" is already enough to back up a whiteboard derivation.)

One more doubt to clear up: the hand-calculation example coming up has only 4 points, [0, 1, 0, -1] — could getting 0.707 just be a coincidence? The next cell checks it in code: **change the frequency, sample rate, and duration however you like, and the RMS stays pinned near 0.707·A** — as long as you cover roughly complete periods, how many points you take doesn't matter.

In [ ]:
# Seeing is believing: the average height of sin² is 1/2, so any pure sine wave has RMS ≈ A/√2
# (This cell doesn't depend on the functions you implement, so you can run it any time)
import numpy as np
import matplotlib.pyplot as plt

t_dense = np.linspace(0, 1, 1000)
s  = np.sin(2 * np.pi * 2 * t_dense)   # a 2 Hz sine wave
s2 = s ** 2                            # its square

fig, axes = plt.subplots(2, 1, figsize=(9, 4.2), sharex=True)
axes[0].plot(t_dense, s, 'C0'); axes[0].axhline(0, color='k', lw=0.5)
axes[0].set_ylabel('sin')
axes[0].set_title('sin: the positive and negative halves cancel, mean ≈ 0 — so the mean cannot measure loudness')
axes[1].plot(t_dense, s2, 'C1'); axes[1].axhline(0.5, color='k', ls='--', lw=1)
axes[1].set_ylabel('sin²'); axes[1].set_xlabel('Time (s)')
axes[1].set_title('sin²: swings between 0 and 1, symmetric about the dashed line at 1/2 → mean = 1/2')
plt.tight_layout(); plt.show()

# Not a 4-point coincidence: change frequency/sample rate/duration however you like and RMS stays pinned near A/√2
print(f'Theoretical value 1/√2 = {1/np.sqrt(2):.4f}')
for freq, sr_i, dur in [(1, 8, 1.0), (3, 100, 1.0), (440, 8000, 0.5), (7, 44100, 0.3)]:
    tt = np.arange(round(dur * sr_i)) / sr_i
    x = np.sin(2 * np.pi * freq * tt)
    rms = np.sqrt(np.mean(x ** 2))
    print(f'  freq={freq:>4} Hz, sr={sr_i:>6}, dur={dur}s → RMS = {rms:.4f}')
print('→ The last row is not a whole number of periods, so it is slightly off; the more complete the periods, the closer to 0.7071.')

## ✏️ Task 3: implement `signal_summary(x)`

Return a dictionary with a statistical summary of the signal:
- `length`: the array's length (an integer)
- `shape`: the array's shape (a tuple)
- `max_abs`: the largest absolute value (a float)
- `mean`: the mean (a float)
- `rms`: the Root Mean Square = √(mean(x²)) (a float)

**By hand** (x = make_sine(1.0, 4, 1.0), i.e. 4 points: [0, 1, 0, -1]):

| Key | Formula | By hand |
|---|---|---|
| length | `len(x)` | ✏️ ? |
| shape | `x.shape` | ✏️ ? |
| max_abs | `max(|0|,|1|,|0|,|-1|)` | ✏️ ? |
| mean | `(0+1+0-1)/4` | ✏️ ? |
| rms | `√((0²+1²+0²+1²)/4)` | ✏️ ? (hint: for a pure sine wave, RMS = A/√2) |

In [ ]:
def signal_summary(x):
    # ✏️ TODO: return a dictionary with the following keys
    #   length  : len(x)
    #   shape   : x.shape
    #   max_abs : np.max(np.abs(x))
    #   mean    : float(np.mean(x))
    #   rms     : float(np.sqrt(np.mean(x**2)))  ← root mean square; whiteboard challenge Q4 uses it
    raise NotImplementedError("TODO: return a dict with length/shape/max_abs/mean/rms")

In [ ]:
try:
    import numpy as np

    x_test = make_sine(1.0, 64, 3.0)
    s = signal_summary(x_test)

    assert s.get('length') == 64,                    f"length should be 64, got {s.get('length')}"
    assert s.get('shape') == (64,),                  f"shape should be (64,), got {s.get('shape')}"
    assert abs(s.get('max_abs', 0) - 1.0) < 0.02,   f"max_abs should be about 1.0"
    assert abs(s.get('mean', 1)) < 0.05,             f"the mean should be close to 0"
    assert abs(s.get('rms', 0) - 1/np.sqrt(2)) < 0.02, f"rms should be about 1/√2 ≈ 0.707"

    # Hand-calculation check: the 4 points [0, 1, 0, -1]
    x4 = make_sine(1.0, 4, 1.0)
    s4 = signal_summary(x4)
    print('Hand-calculation check (4 points [0, 1, 0, -1]):')
    print(f'  x4     = {np.round(x4, 3)}')
    print(f'  summary = {s4}')
    print()
    print('signal_summary  OK')
    print(f'  rms measured={s["rms"]:.4f}  theoretical 1/√2={1/np.sqrt(2):.4f}')
except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO function above first, then run this demo cell')

### 📦 Side box: by hand it's 0, but the screen says 5.551115123125783e-17 — you didn't get it wrong!

In the previous cell's printout, the `mean` entry most likely isn't a clean 0 but a long string like `5.551115123125783e-17`. If you're starting to suspect your function is wrong — stop, and look at two things first:

**First: how to read this notation.** `e-17` is the keyboard way of writing **scientific notation**: `5.55e-17` = 5.55 × 10⁻¹⁷ = 0.0000000000000000555 (16 zeros after the decimal point, then 555). That's smaller than a speck of dust compared with the whole Earth — for all practical purposes, it's 0.

**Second: why it isn't exactly 0.** Computers store decimals in binary, and many decimal fractions simply can't be stored "exactly" (just as 1/3 never ends in decimal: 0.3333…), so every operation carries a **tiny rounding error**. The exact 0 you got by hand becomes "a number only 10⁻¹⁷ away from 0" inside the computer.

**Conclusion: when you see magnitudes like e-15, e-16, e-17, read them as 0 — that's actually a sign you got it right.**

This also explains how every answer-checking assertion in this lesson is written: always `abs(...) < a small tolerance` or `np.isclose`, never `==`. When your hand calculation and the code "check answers," what they're checking is "close enough," not "identical down to the last digit."

While we're at it, let's defuse two more scares of the same kind so you don't panic when you see them later:
- An occasional `-0.` in a printout ("negative zero"?!): it's just a 0 that was rounded from the negative side, carrying a negligible minus sign;
- In the whiteboard challenge later on, if the exact answer you derive for Q3 is 0, the program will print `-0.000000` or `-1.5e-15` — same thing; you got it right.

The next cell lets you poke at this error yourself.

In [ ]:
# Poke at floating-point error yourself (this cell doesn't depend on the functions you implement, so you can run it any time)
import numpy as np

print('0.1 + 0.2 =', repr(0.1 + 0.2), '  ← not 0.3, believe it or not!')
print('0.1 + 0.2 == 0.3 ?          ', 0.1 + 0.2 == 0.3)
print('np.isclose(0.1 + 0.2, 0.3) ?', np.isclose(0.1 + 0.2, 0.3))
print()
print('sin(π) should be 0 in theory; the computer gives:', np.sin(np.pi), ' ← on the order of e-16, read it as 0')
print()
print('→ So always "check answers" with np.isclose / a tolerance, never with ==.')

## Three NumPy essentials: dtype / broadcasting / slicing

The DSP code in L32–L51 leans heavily on three NumPy mechanisms. Here we demo them on sound signals, laying the groundwork for what comes later.

**First, two words to clear up: bit and byte**

At the lowest level, a computer is nothing but "switches": on = 1, off = 0. One switch is 1 **bit**. Eight switches packed together make 1 **byte** — the basic unit of memory, like a "dozen" for eggs (1 dozen = 12).

With those two words in hand, the number in a dtype's name tells you its cost directly:

- float**64**: 64 switches to store one decimal number → 64 ÷ 8 = **8 bytes**
- float**32**: 32 switches to store one decimal number → 32 ÷ 8 = **4 bytes**

More switches means finer-grained decimals, but the memory used grows right along with them. The next cell uses three new terms, so here's a mini-dictionary first:

| Notation | Meaning |
|---|---|
| `arr.itemsize` | how many bytes **each element** of the array takes |
| `arr.nbytes` | how many bytes **the whole array** takes in total (= itemsize × number of elements) |
| `arr.astype(np.float32)` | store the array again at a different precision (64 switches → 32 switches) |


In [ ]:
try:
    # ── dtype: float64 vs float32 ──────────────────────────────────────
    # The two precisions common in audio processing
    a64 = np.zeros(8, dtype=np.float64)
    a32 = np.zeros(8, dtype=np.float32)
    print(f'float64: {a64.itemsize} bytes per element; float32: {a32.itemsize} bytes per element')
    print(f'float64 uses {a64.nbytes} bytes  →  float32 uses {a32.nbytes} bytes (half the memory)')

    # make_sine produces float64 by default; converting to float32:
    t = make_time_axis(0.5, 8)
    x64 = np.sin(2 * np.pi * 1.0 * t)           # float64, the default
    x32 = x64.astype(np.float32)                  # float32, common in deep learning
    print(f'\nmake_time_axis → dtype={x64.dtype},  after converting to float32, dtype={x32.dtype}')
    print(f'Precision difference: max |x64 - x32| = {np.abs(x64 - x32.astype(np.float64)).max():.2e}')
    # Why does audio usually use float32? Half the memory, native GPU/neural-network support, and more than enough precision for human ears.

except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO function above first, then run this demo cell')

In [ ]:
try:
    # ── broadcasting: adding two sine waves together ────────────────────────
    # Scalar times array: NumPy automatically broadcasts the scalar to every element
    t = make_time_axis(1.0, 16)
    wave_A = 0.7 * np.sin(2 * np.pi * 1.0 * t)  # 0.7 is a scalar, broadcast to all 16 elements
    wave_B = 0.3 * np.sin(2 * np.pi * 3.0 * t)

    # Adding the two signals: shape (16,) + shape (16,) → shape (16,)
    mixed = wave_A + wave_B
    print(f'wave_A.shape={wave_A.shape}, wave_B.shape={wave_B.shape} → mixed.shape={mixed.shape}')

    # Stacking several signals into a matrix (rows = channels, columns = time)
    multi = np.stack([wave_A, wave_B, mixed])      # shape (3, 16)
    print(f'after stacking, multi.shape={multi.shape}  (3 channels × 16 samples)')
    assert mixed.shape == (16,)
    assert multi.shape == (3, 16)
    print('✓ Broadcasting and stacking shapes are correct')

except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO function above first, then run this demo cell')

In [ ]:
try:
    # ── slicing: cutting out a section and downsampling ───────────────────────────────
    sr_demo, dur_demo = 8, 2.0                     # 8 Hz, 2 seconds, 16 points in total
    x = make_sine(dur_demo, sr_demo, 1.0)
    print(f'full signal x.shape={x.shape}  →  {x.round(2)}')

    # Slice 1: cut out second 1 (indices sr : 2*sr)
    x_sec1 = x[sr_demo : 2 * sr_demo]
    print(f'second 1  x[{sr_demo}:{2*sr_demo}].shape={x_sec1.shape}  →  {x_sec1.round(2)}')

    # Slice 2: downsampling (keep every other point → halves the sample rate)
    x_down = x[::2]
    print(f'downsampled  x[::2].shape={x_down.shape}  →  {x_down.round(2)}')

    assert x_sec1.shape == (sr_demo,), f'expected ({sr_demo},), got {x_sec1.shape}'
    assert x_down.shape == (sr_demo,), f'expected ({sr_demo},), got {x_down.shape}'
    print('✓ Slice shapes are correct')

except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO function above first, then run this demo cell')

## Aurora connection

**The good news first**: today's four functions are your very own **toolbox** — implement them in the notebook first, then pin them down with asserts; later, L32–L37 just keep carrying the same intuition forward.

They line up **conceptually** with the frequency-domain pipeline in the Aurora Audio Core, but they are **not** ready-made entry points in `transforms.py` (the repo's `transforms.py` holds the frequency-domain kernels such as dft/fft/stft, with different interfaces):

| This lesson's function | Conceptual counterpart | Notes |
|---|---|---|
| `samples_count` | frame length / number of samples | the first step of every signal function |
| `make_time_axis` | `t[n]=n/sr` | L32 builds the time axis again |
| `make_sine` | sine wave generation | used again and again in L33–L37 |
| `signal_summary` | signal statistics summary | L40 and L51 read RMS and similar quantities |

> So don't write `from aurora.audio.transforms import make_sine` — there's no student-version entry point for it there.
> L32 (`notebooks/5_audio_dsp/L32_numpy_signals.ipynb`) reuses this lesson's intuition to build "sample points + time coordinates" into a complete foundation for signals.


## ✏️ Whiteboard challenge: no code, derive it on paper (target: 10 minutes)

This is your closed-book self-test once you've finished the lesson. **Cover the screen** and work through the following derivations on paper:

---

**Problem**: the signal's parameters are
- sample rate `sr = 8000` Hz (telephone quality)
- duration `duration = 0.5` seconds
- frequency `freq = 440` Hz (the A4 note)
- amplitude $A = 0.7$ (parameter name in the code: `amplitude`)

**Question 1**: how many elements does the array `x` have? (Write out the formula.)

**Question 2**: what is `t[100]`, in seconds? (`t` is the time-axis array.)

**Question 3**: what is the value of `x[100]`? (Express it with `sin`; no need to compute an exact value.)

**Question 4**: what is the theoretical value of `signal_summary(x)['rms']`?
Hint: the RMS of a pure sine wave `A·sin(...)` is A / √2 (why √2? Go back to the section before Task 3 on "the average of sin² is 1/2").

---

Once you've finished deriving, run the check cell below to compare answers.

> **The Q3 Easter egg (read it after you've derived your answer)**: if you carry the angle all the way through — 2π × 440 × 0.0125 = 11π, and sin is exactly 0 at every integer multiple of π — then your paper answer is exactly 0. Is the check cell printing something like `-0.000000` or `-1.5e-15`? That's the floating-point version of 0 (see the side box after Task 3): it means you derived it correctly — more precisely than the computer, in fact.

In [ ]:
try:
    # ✏️ Answer-check cell — finish the whiteboard derivation before you run it
    import numpy as np

    sr, duration, freq, amplitude = 8000, 0.5, 440, 0.7

    # Q1: number of elements
    N_expected = round(duration * sr)
    t = make_time_axis(duration, sr)
    x = make_sine(duration, sr, freq, amplitude)
    assert len(x) == N_expected, f"Q1: expected {N_expected}, got {len(x)}"
    print(f"Q1 ✅  N = round({duration} × {sr}) = {N_expected} samples")

    # Q2: t[100]
    t100_expected = 100 / sr
    assert np.isclose(t[100], t100_expected, atol=1e-10), f"Q2: expected {t100_expected:.6f}, got {t[100]:.6f}"
    print(f"Q2 ✅  t[100] = 100 / {sr} = {t100_expected:.6f} s")

    # Q3: x[100]
    x100_expected = amplitude * np.sin(2 * np.pi * freq * t[100])
    assert np.isclose(x[100], x100_expected, atol=1e-10)
    print(f"Q3 ✅  x[100] = {amplitude} × sin(2π × {freq} × {t100_expected:.6f}) = {x100_expected:.6f}")

    # Q4: theoretical RMS
    rms_theory = amplitude / np.sqrt(2)
    s = signal_summary(x)
    assert np.isclose(s['rms'], rms_theory, atol=0.01), f"Q4: theory {rms_theory:.4f}, measured {s['rms']:.4f}"
    print(f"Q4 ✅  RMS ≈ {amplitude}/√2 = {rms_theory:.4f}  measured {s['rms']:.4f}")
    print()
    print("🎉 Whiteboard challenge passed! You've internalized the math of sampling, time axes, and sine waves.")
except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO function above first, then run this demo cell')

## Wrap-up

| Function | What it does | Where it shows up later |
|---|---|---|
| `samples_count` | N = duration × sr | the first step of every signal function |
| `make_time_axis` | builds the time axis | L32 numpy_signals, L33 sine_wave |
| `make_sine` | generates a sine wave | L34 aliasing, L36 windows, L37 DFT |
| `signal_summary` | summarizes statistics | L40 spectrum, L51 real_audio |

**The method this lesson used**: fill in the derivation table by hand first, then write code to check the answers.
This method runs through the entire Aurora course — every step of the FFT will also be worked out by hand before it's implemented.

**Key intuition**: sound = a 1D float64 array, with length = duration × sample_rate.
The FFT transforms this array into the frequency domain — that's for L37 and beyond.

**Next lesson L03**: we're still not learning "how to compute" the FFT — first, you'll learn to **read** a time–frequency plot (spectrogram).
You'll see the time–frequency plots of pure tones / chords / sweeps / noise, planting the visual intuition for L37–L41.

In [ ]:
# ✏️ Self-review for this lesson
l02_review = {
    "samples_count_done":   None,  # samples_count implemented and passing its asserts? True/False
    "make_time_axis_done":  None,  # make_time_axis implemented and passing its asserts? True/False
    "make_sine_done":       None,  # make_sine implemented and passing its asserts? True/False
    "signal_summary_done":  None,  # signal_summary implemented and passing its asserts? True/False
    "whiteboard_passed":    None,  # whiteboard challenge (derivation on paper) done? True/False
    "sampling_rate_intuition": None,  # can I explain why sr=44100 and not 44000? True/False
    #   ↑ two questions in one: (a) why it has to be > 40000 (Nyquist: at least 2 points per period);
    #     (b) why exactly 44100 (history: video recorder tape was borrowed to store audio, and NTSC and PAL both work out to 44100).
    #     Both answers are in Section 1, "Follow-up 1 / Follow-up 2"; if you can't answer, just go back and reread those two subsections.
}

unfilled = [k for k, v in l02_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l02_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
    print('   Suggestion: reread the matching sections, and check your understanding by hand rather than by running code.')
else:
    print('✅ L02 fully cleared! On to L03: spectrogram intuition')

---

→ **Next lesson** · [L03 · Spectrogram intuition](L03_spectrogram.en.ipynb)

> Next time: **spectrogram intuition** — learning to read the three axes of a time–frequency plot before you learn the FFT.